# Финансовый анализ, стресс-тестирование и машинное обучение

Импорт необходимых библиотек для работы с данными и вычислений, инструментов машинного обучения, а также библиотеки градиентного бустинга `CatBoost`. Загрузка готового исторического датасета `full_bonds_dataset.csv` для старта финансового анализа и построения предиктивных моделей.

In [1]:
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split


In [ ]:
df = pd.read_csv("notebooks/rate_sensitivity/bond_shock/full_bonds_dataset.csv")

Приведение полей с датами сделки и погашения к формату `datetime` для проведения дальнейших расчетов. Фильтрация датасета с удалением строк, содержащих пропуски в ключевых финансовых показателях, необходимых для расчета дюрации и корректного обучения моделей.

In [3]:
df["TRADEDATE"] = pd.to_datetime(df["TRADEDATE"])
df["MATDATE"] = pd.to_datetime(df["MATDATE"])
df = df.dropna(
    subset=["CLOSE", "DURATION", "YIELDCLOSE", "MATDATE", "FACEVALUE"]
).copy()

Вычисление временных и рисковых параметров для каждой облигации на каждый день торгов:
1. **Срок до погашения:** расчет остатка срока в годах с отсечением коротких бумаг.
2. **Модифицированная дюрация:** пересчет дюрации из дней в года с дисконтированием по текущей доходности. Фильтрация аномальных значений.
3. **Выпуклость:** расчет приближенной выпуклости для моделирования нелинейного изменения цены при сильных шоках доходности. Вывод итогового количества валидных биржевых записей.

In [4]:
df["T"] = (df["MATDATE"] - df["TRADEDATE"]).dt.days / 365.25
df = df[df["T"] > 0.05]

In [5]:
df["DurationMod"] = (df["DURATION"] / 365.0) / (1 + df["YIELDCLOSE"] / 100.0)
df = df[(df['DurationMod'] > 0) & (df['DurationMod'] < 30)].copy()

In [6]:
y = df["YIELDCLOSE"] / 100.0
df["Convexity"] = (df["T"] * (df["T"] + 1)) / ((1 + y) ** 2) * 0.75

In [7]:
print(f"Базовых строк облигаций после фильтрации: {len(df)}")

Базовых строк облигаций после фильтрации: 78865


Ограничение выборки для исключения аномальных и нерепрезентативных значений:
1. **Номинал:** отбор облигаций с положительным номиналом не превышающим 1 000 000 рублей.
2. **Модифицированная дюрация:** сужение диапазона до разумных рыночных пределов от 0 до 20 лет для исключения ультрадлинных или ошибочных инструментов.

In [8]:
df = df[(df["FACEVALUE"] > 0) & (df["FACEVALUE"] <= 1000000)].copy()
df = df[(df["DurationMod"] > 0) & (df["DurationMod"] <= 20)].copy()

Создание сетки параллельных сдвигов кривой доходности в базисных пунктах (`shocks_bp` от -200 до +200 б.п.). Для каждого шокового сценария формируется копия базового датасета с добавлением двух ключевых колонок:
1. **`shock_bp`** - величина шока в базисных пунктах.
2. **`shock_abs`** - абсолютное изменение ставки в долях единицы, используемое для дальнейших расчетов.

Все индивидуальные сценарии объединяются с помощью `pd.concat` в единый сводный датафрейм `df_scenarios` для последующего стресс-тестирования портфеля облигаций.

In [9]:
shocks_bp = [-200, -150, -100, -50, -25, 25, 50, 100, 150, 200]
scenarios_list = []
for s_bp in shocks_bp:
    temp_df = df.copy()
    temp_df["shock_bp"] = s_bp
    temp_df["shock_abs"] = (
        s_bp / 10000.0
    )
    scenarios_list.append(temp_df)

df_scenarios = pd.concat(scenarios_list, ignore_index=True)

Оценка реакции стоимости облигаций на заданные параллельные сдвиги кривой доходности с использованием аппроксимации Тейлора второго порядка (модифицированная дюрация и выпуклость):

1. **Относительное изменение цены:** расчет процентного сдвига котировок по формуле.
2. **Денежное изменение портфеля:** пересчет относительного изменения в абсолютные рубли для фиксированного размера позиции с учетом номинала.
3. Вывод общего размера сгенерированного датасета сценариев и предварительный просмотр первых строк по ключевым метрикам.

In [10]:
dr = df_scenarios["shock_abs"]
d_mod = df_scenarios["DurationMod"]
conv = df_scenarios["Convexity"]

df_scenarios["price_change_pct"] = -d_mod * dr + 0.5 * conv * (dr**2)
position_size = 10
df_scenarios["target_rub_change"] = df_scenarios["price_change_pct"] * (
    df_scenarios["FACEVALUE"] * position_size
)

print(f"Размер: {df_scenarios.shape[0]} строк.")
df_scenarios[["SECID", "TRADEDATE", "CLOSE", "shock_bp", "DurationMod", "target_rub_change"]].head(3)

Размер: 783130 строк.


,SECID,TRADEDATE,CLOSE,shock_bp,DurationMod,target_rub_change
0,RU000A0JTU85,2014-01-09,102.40,-200,4.559859,1190.155642
1,RU000A0JTU85,2014-01-14,102.04,-200,4.543787,1186.011149
2,RU000A0JTU85,2014-01-15,101.88,-200,4.537449,1184.486013


Разделение датасета сценариев на обучающую и тестовую выборки по уникальным идентификаторам облигаций в пропорции 80/20:

1. **Защита от утечки данных:** Разделение выполняется по уникальным тикерам, а не по случайным строкам. Это гарантирует, что модель будет тестироваться на абсолютно «невиданных» ей выпусках облигаций, оценивая ее генерализующую способность.
2. Фильтрация датасета сценариев `df_scenarios` для получения изолированных подмножеств.
3. Вывод итоговой статистики: количество уникальных облигаций и размер датасета (количество строк со стресс-сценариями) для train и test выборок.

In [11]:
unique_secids = df_scenarios["SECID"].unique()

train_secids, test_secids = train_test_split(
    unique_secids, test_size=0.2, random_state=42
)
train_df = df_scenarios[df_scenarios["SECID"].isin(train_secids)].copy()
test_df = df_scenarios[df_scenarios["SECID"].isin(test_secids)].copy()

print(f"Уникальных облигаций в Train: {len(train_secids)}")
print(f"Уникальных облигаций в Test: {len(test_secids)}")
print(f"Размер обучающей выборки (строк со сценариями): {train_df.shape[0]}")
print(f"Размер тестовой выборки (строк со сценариями): {test_df.shape[0]}")


Уникальных облигаций в Train: 114
Уникальных облигаций в Test: 29
Размер обучающей выборки (строк со сценариями): 613810
Размер тестовой выборки (строк со сценариями): 169320


Формирование обучающих и тестовых наборов данных для базовых моделей линейной и квадратичной регрессии:
1. **Линейные признаки:** Включают только абсолютную величину шока для оценки простой линейной чувствительности.
2. **Квадратичные признаки:** Содержат сам шок и его квадрат, что позволяет модели приблизиться к учету нелинейных эффектов (выпуклости).
3. **Целевая переменная:** Абсолютное денежное изменение стоимости позиции в рублях, выступающее в роли таргета для обучения.

In [12]:
X_train_lin = train_df[["shock_abs"]]
X_test_lin = test_df[["shock_abs"]]

X_train_quad = pd.DataFrame(
    {"shock": train_df["shock_abs"], "shock_sq": train_df["shock_abs"] ** 2}
)
X_test_quad = pd.DataFrame(
    {"shock": test_df["shock_abs"], "shock_sq": test_df["shock_abs"] ** 2}
)

y_train = train_df["target_rub_change"]
y_test = test_df["target_rub_change"]

In [13]:
lin_reg = LinearRegression()
lin_reg.fit(X_train_lin, y_train)
pred_lin = lin_reg.predict(X_test_lin)

quad_reg = LinearRegression()
quad_reg.fit(X_train_quad, y_train)
pred_quad = quad_reg.predict(X_test_quad)

Определение вспомогательной функции `evaluate_metrics` для расчета ключевых метрик качества регрессии (MAE и RMSE) в абсолютном денежном выражении. Запуск сравнительной оценки базовых аналитических моделей - линейной чувствительности и квадратичной модели с учетом выпуклости - на независимой тестовой выборке.

In [14]:
pred_analytical = test_df[
    "target_rub_change"
]

def evaluate_metrics(y_true, y_pred, name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    print(f"[{name}]")
    print(f"  MAE  = {mae:.4f} руб.")
    print(f"  RMSE = {rmse:.4f} руб.\n")


print("Результаты оценки на тестовой выборке:")
evaluate_metrics(y_test, pred_lin, "Линейная модель (только шок)")
evaluate_metrics(y_test, pred_quad, "Квадратичная модель (+ учет выпуклости)")

Результаты оценки на тестовой выборке:
[Линейная модель (только шок)]
  MAE  = 746.5973 руб.
  RMSE = 759.2325 руб.

[Квадратичная модель (+ учет выпуклости)]
  MAE  = 747.4850 руб.
  RMSE = 1047.4861 руб.



Построение и обучение модели машинного обучения для прогнозирования изменения стоимости портфеля с учетом индивидуальных характеристик выпусков:

1. **Формирование набора признаков:** Объединение сценарного макрофактора и ключевых параметров облигаций, что позволяет модели учитывать рыночную специфику конкретных бумаг.
2. **Настройка регрессора:** Инициализация модели с функцией потерь абсолютной ошибки, фиксированной глубиной деревьев и темпом обучения.
3. **Обучение с ранней остановкой:** Использование тестовой выборки в качестве валидационного набора для предотвращения переобучения.
4. Генерация предсказаний и сравнительная оценка точности CatBoost на независимом тесте.

In [15]:
feature_cols = ["shock_abs", "DurationMod", "YIELDCLOSE", "FACEVALUE"]

X_train_cb = train_df[feature_cols]
X_test_cb = test_df[feature_cols]

cb_model = CatBoostRegressor(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    loss_function="MAE",
    verbose=100,
    random_seed=42,
)

cb_model.fit(
    X_train_cb, y_train, eval_set=(X_test_cb, y_test), early_stopping_rounds=50
)

pred_cb = cb_model.predict(X_test_cb)

print("\nРезультаты CatBoost на тестовой выборке:")
evaluate_metrics(y_test, pred_cb, "CatBoost (с учетом фичей облигации)")


0:	learn: 924.1966948	test: 171.2828703	best: 171.2828703 (0)	total: 186ms	remaining: 1m 32s
100:	learn: 761.2455322	test: 10.8579445	best: 10.8579445 (100)	total: 5.05s	remaining: 20s
200:	learn: 726.7635727	test: 7.6163250	best: 7.6163250 (200)	total: 8.41s	remaining: 12.5s
300:	learn: 693.4769042	test: 6.7143710	best: 6.7143710 (300)	total: 11.7s	remaining: 7.76s
400:	learn: 691.9818070	test: 6.1783162	best: 6.1783162 (400)	total: 15.1s	remaining: 3.73s
499:	learn: 650.9086870	test: 5.8734859	best: 5.8734859 (499)	total: 18.5s	remaining: 0us

bestTest = 5.873485869
bestIteration = 499


Результаты CatBoost на тестовой выборке:
[CatBoost (с учетом фичей облигации)]
  MAE  = 5.8735 руб.
  RMSE = 14.9784 руб.



Детальная оценка стабильности работы модели CatBoost при различных сценариях стресс-тестирования:

1. **Расчет пообъектных ошибок:** Добавление предсказаний модели к тестовой выборке и вычисление абсолютной ошибки в рублях для каждого наблюдения.
2. **Агрегация по величине шока:** Группировка данных по каждому сценарию сдвига кривой доходности для проверки устойчивости и выявления возможных зон роста ошибки.
3. **Расчет метрик:** Вычисление среднего значения (MAE), медианы и максимальной абсолютной ошибки для каждого уровня шока с последующим выводом итоговой сводной таблицы.

In [16]:
test_eval = test_df.copy()
test_eval["pred_cb"] = pred_cb
test_eval["error_cb"] = np.abs(test_eval["target_rub_change"] - test_eval["pred_cb"])

shock_analysis = (
    test_eval.groupby("shock_bp")["error_cb"]
    .agg(["mean", "median", "max"])
    .reset_index()
)
shock_analysis.rename(
    columns={"mean": "MAE_rub", "median": "Median_Error_rub", "max": "Max_Error_rub"},
    inplace=True,
)

print("Ошибка CatBoost в зависимости от размера процентного шока:")
print(shock_analysis.to_string(index=False))

Ошибка CatBoost в зависимости от размера процентного шока:
 shock_bp   MAE_rub  Median_Error_rub  Max_Error_rub
     -200 11.301827          5.745386    2487.431465
     -150  8.328173          4.850141    1403.503019
     -100  4.932993          2.979780     619.294899
      -50  2.977215          2.052403     153.024858
      -25  1.310016          0.762652      37.239903
       25  1.952226          1.122843      37.994304
       50  2.458577          1.576525     151.611502
      100  4.522419          2.997963     626.016291
      150  9.229067          4.921698    1411.334054
      200 11.722354          5.619744    2505.587012


C:\Users\dkita\AppData\Local\Temp\ipykernel_25348\1260386940.py:10: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  shock_analysis.rename(


## Итоги этапа: финансовое моделирование и стресс-тестирование

В рамках данного ноутбука был успешно реализован полный цикл аналитического и предиктивного моделирования для оценки чувствительности портфеля облигаций:

1. **Финансовый инжиниринг:** На основе исторических биржевых данных рассчитаны ключевые показатели: срок до погашения, модифицированная дюрация и выпуклость.
2. **Стресс-тестирование:** Сгенерирована сетка из 10 параллельных сценариев шоков кривой доходности с расчетом процентного и рублевого изменения стоимости позиций.
3. **Валидация:** Выполнено разделение выборки по уникальным тикерам, что исключило утечку данных и обеспечило честное тестирование моделей на новых облигациях.
4. **Сравнение подходов:** Проведено обучение базовых аналитических моделей (линейная регрессия и учет выпуклости) и современного алгоритма градиентного бустинга.

### Анализ результатов и сравнение метрик

* **Базовые модели (Линейная и Квадратичная):** 
  * Показали погрешность: **MAE: 746–747 руб.** при размере позиции в 10 штук. 
  * Использование квадратичной аппроксимации (с учетом выпуклости) не дало прироста точности по MAE, а по RMSE метрика выросла (до 1047 руб. против 759 руб. у линейной), что говорит о чувствительности классических формул к выбросам на крупных шоках.

* **Модель CatBoost:**
  * Продемонстрировала превосходство: **MAE** снизилась до 5.87 руб., а **RMSE** — до 14.98 руб.
  * Учет дополнительных параметров облигаций в сочетании с нелинейной структурой деревьев позволил модели практически идеально восстановить целевую зависимость.

* **Анализ стабильности по сценариям шоков:**
  * Ошибка предсказания CatBoost ожидаемо симметрично растет по мере удаления величины шока от нуля.
  * Модель сохраняет высокую устойчивость, а медианные значения ошибок остаются минимальными во всех диапазонах.

### **Общий вывод:** Построенный пайплайн машинного обучения решает задачу оценки рыночного риска. Градиентный бустинг превзошел классические расчетные формулы, обеспечив высокую точность прогноза изменения стоимости портфеля в рублях.